# LoRA fine-tuning for behavior: formal tone + strict JSON

This notebook mirrors `train_lora.py` and `evaluate.py` step by step, so you can run everything in Google Colab (free T4 GPU) or locally.

**The idea.** We fine-tune a small chat model for *behavior*, not for *facts*:

- answer in a formal, compliance-approved tone,
- always return one strict JSON object (`answer`, `tone`, `sources`, `needs_escalation`),
- use **only** the context in the prompt, and say so (with `needs_escalation: true`) when the context does not contain the answer.

The company policies in the data are fictional ("Globex Corp"), and every prompt carries its own context. Nothing about the policies needs to live in the weights. The training passages are deliberately different from the RAG starter's corpus (a different fictional company and different topics), so nothing the model memorizes can conflict with retrieved facts. For facts that change, use retrieval (RAG); use fine-tuning for how the model should *behave*.

**Before you start (Colab).** In Colab use File > Upload notebook and upload `lora_finetune.ipynb`, then Runtime > Change runtime type > T4 GPU. Step 2 asks you to upload the three data files. On a CPU this still works for a 0.5B model but is slow.

> Honest note: the scripts in this repository were smoke-tested with tiny models and run end to end on CPU with the default model (Qwen2.5-0.5B-Instruct); the measured numbers are in `README.md`. This notebook's code cells were executed locally (outside Colab) against a tiny random model, only to check that they run; they were not run in Colab or on a GPU. Hyper-parameters below are starting points, not tuned values.

## Step 0: install

Colab already ships PyTorch. Locally, install PyTorch first (see `requirements.txt`) and skip this cell if the libraries are already installed. If Colab asks you to restart the runtime after the install, do that and then continue from Step 1.

In [ ]:
%pip install -q "transformers>=4.51" "peft>=0.14" "datasets>=3.0" "accelerate>=1.0"

## Step 1: settings and hardware check

Everything you may want to change is in this one cell. Try changing `EPOCHS` or `LEARNING_RATE` and compare the loss curves in Step 6.

In [ ]:
import inspect
import json
import math
import re
from pathlib import Path

import peft
import torch
import transformers

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"   # any small chat model that has a chat template
EPOCHS = 4
LEARNING_RATE = 2e-4
LORA_RANK, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
BATCH_SIZE, GRAD_ACCUM = 4, 2
MAX_LEN = 512          # examples longer than this many tokens are dropped, never cut in half
SEED = 42
OUT_DIR = Path("outputs/lora-adapter")

print("transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU: works, but slowly)")

## Step 2: load the data

Three small JSONL files live in the repo's `data/` folder:

- `train.jsonl` (about 70 examples) and `val.jsonl` (about 24, on *different* passages, held out),
- `general_probe.jsonl` (10 generic questions used later to check for catastrophic forgetting).

If you opened this notebook in Colab without the repo, the cell asks you to upload those three files.

In [ ]:
DATA_DIR = Path("data")
if not (DATA_DIR / "train.jsonl").exists():
    try:  # Google Colab: upload the three files from the repo's data/ folder
        from google.colab import files
        DATA_DIR.mkdir(exist_ok=True)
        print("Select train.jsonl, val.jsonl and general_probe.jsonl")
        for name, content in files.upload().items():
            (DATA_DIR / name).write_bytes(content)
    except ImportError:
        raise RuntimeError("Run this notebook from the finetune-starter folder so that data/train.jsonl exists.")


def read_jsonl(path):
    with open(path, encoding="utf-8-sig") as f:
        return [json.loads(line) for line in f if line.strip()]


train_rows = read_jsonl(DATA_DIR / "train.jsonl")
val_rows = read_jsonl(DATA_DIR / "val.jsonl")
probe_rows = read_jsonl(DATA_DIR / "general_probe.jsonl")
print(len(train_rows), "train |", len(val_rows), "val |", len(probe_rows), "probe prompts")

### Look at one example

Each example is a chat: a short system prompt, a user turn with a `CONTEXT` (fictional policy text with an id in brackets) plus a `QUESTION`, and the assistant turn, which is **one JSON object**. About 28 percent of the examples are *unanswerable*: the context does not contain the answer, so the target says so and sets `needs_escalation` to `true`. The hope is that the model learns to abstain instead of guessing; whether a small model really does is something Step 7 measures (the README reports a real run).

In [ ]:
def show(messages):
    for msg in messages:
        print(f"--- {msg['role'].upper()} ---")
        content = msg["content"]
        if msg["role"] == "assistant":
            content = json.dumps(json.loads(content), indent=2)
        print(content)


show(train_rows[0]["messages"])
print("\n\n=========== an UNANSWERABLE example ===========")
unanswerable = next(r for r in train_rows if json.loads(r["messages"][-1]["content"])["needs_escalation"])
show(unanswerable["messages"])

## Step 3: tokenizer, chat template and loss masking

Two details matter here:

1. **Chat template.** Instruct models expect a specific text layout (special tokens around each turn). `tokenizer.apply_chat_template` produces it. If a tokenizer has no template we fall back to a plain `### System / ### User / ### Assistant` layout.
2. **Loss masking.** We set the label of every prompt token (system + user + the assistant header) to `-100`, which the loss ignores. The model is trained *only* on the assistant turn: the JSON plus the chat template's end-of-turn token and trailing newline. Without this, the model would also be graded on reproducing the policy text and the question, which wastes capacity and blurs the behavior we want.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
USE_TEMPLATE = bool(getattr(tokenizer, "chat_template", None))
IGNORE = -100
print("chat template available:", USE_TEMPLATE)


def plain_prompt(messages):
    parts = [f"### {m['role'].capitalize()}:\n{m['content']}\n" for m in messages]
    return "\n".join(parts + ["### Assistant:\n"])


def render_prompt(messages):
    if USE_TEMPLATE:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return plain_prompt(messages)


def encode(text, add_special_tokens):
    return tokenizer(text, add_special_tokens=add_special_tokens)["input_ids"]


def encode_prompt(messages):
    # chat-template text already contains its special tokens; do not add them twice
    return encode(render_prompt(messages), add_special_tokens=not USE_TEMPLATE)


def build_example(messages, max_len=MAX_LEN):
    prompt_messages, reply = messages[:-1], messages[-1]["content"]
    prompt_ids = encode_prompt(prompt_messages)
    if USE_TEMPLATE:
        prompt_text = render_prompt(prompt_messages)
        full_text = tokenizer.apply_chat_template(messages, tokenize=False)
        if full_text.startswith(prompt_text):
            completion_text = full_text[len(prompt_text):]
        else:
            completion_text = reply + (tokenizer.eos_token or "")
        completion_ids = encode(completion_text, add_special_tokens=False)
    else:
        completion_ids = encode(reply, add_special_tokens=False)
        if tokenizer.eos_token_id is not None:
            completion_ids = completion_ids + [tokenizer.eos_token_id]   # teach the model to stop
    input_ids = prompt_ids + completion_ids
    if len(input_ids) > max_len:
        return None
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": [IGNORE] * len(prompt_ids) + completion_ids,
    }


ex = build_example(train_rows[0]["messages"])
n_masked = sum(1 for x in ex["labels"] if x == IGNORE)
print(f"{len(ex['input_ids'])} tokens in total: {n_masked} masked prompt tokens, {len(ex['input_ids']) - n_masked} trained tokens\n")
print("MASKED (not trained on):\n" + tokenizer.decode(ex["input_ids"][:n_masked]))
print("\nTRAINED ON (the assistant reply):\n" + tokenizer.decode([x for x in ex["labels"] if x != IGNORE]))

In [ ]:
from datasets import Dataset


def encode_split(rows, name):
    examples = [build_example(r["messages"]) for r in rows]
    kept = [e for e in examples if e is not None]
    print(f"{name}: kept {len(kept)} of {len(rows)} examples")
    return Dataset.from_list(kept)


train_ds = encode_split(train_rows, "train")
val_ds = encode_split(val_rows, "val")

pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else (tokenizer.eos_token_id or 0)


def collate(features):
    longest = max(len(f["input_ids"]) for f in features)

    def pad(key, value):
        return torch.tensor([list(f[key]) + [value] * (longest - len(f[key])) for f in features])

    return {"input_ids": pad("input_ids", pad_id), "attention_mask": pad("attention_mask", 0), "labels": pad("labels", IGNORE)}

## Step 4: load the base model and attach LoRA

LoRA freezes the base weights and learns small low-rank update matrices next to them. `target_modules="all-linear"` puts adapters on every linear layer (except the output head). `r` (rank) is the size of the update: bigger means more capacity and more risk of overfitting. `lora_alpha / r` scales the update.

We load the model in float32 so behavior is the same on every transformers version; newer versions otherwise follow the checkpoint's stored dtype. Mixed precision is switched on in Step 5.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, set_seed

set_seed(SEED)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).float()
model.config.use_cache = False
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # a small fraction of all parameters

## Step 5: train

We evaluate on `val.jsonl` at the end of every epoch so you can watch for overfitting. Two version differences are handled by looking at the installed signatures: `evaluation_strategy` was renamed `eval_strategy`, and `Trainer(tokenizer=...)` became `Trainer(processing_class=...)`. Only the adapter is trained; no checkpoints are written.

In [ ]:
from transformers import Trainer, TrainingArguments


def first_supported(names, *candidates):
    for c in candidates:
        if c in names:
            return c
    raise RuntimeError(f"none of {candidates} exist in this transformers version")


steps_per_epoch = max(1, math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM)))
warmup_steps = max(1, round(0.1 * steps_per_epoch * EPOCHS))
use_cuda = torch.cuda.is_available()
bf16 = use_cuda and torch.cuda.get_device_capability(0)[0] >= 8   # Ampere or newer
fp16 = use_cuda and not bf16                                       # e.g. the free Colab T4

args_kwargs = dict(
    output_dir=str(OUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    weight_decay=0.0,
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    bf16=bf16,
    fp16=fp16,
    dataloader_pin_memory=use_cuda,
    remove_unused_columns=False,
    label_names=["labels"],   # PEFT hides the model signature; name the labels explicitly
)
args_kwargs[first_supported(inspect.signature(TrainingArguments.__init__).parameters, "eval_strategy", "evaluation_strategy")] = "epoch"

trainer_kwargs = dict(
    model=model,
    args=TrainingArguments(**args_kwargs),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate,
)
trainer_kwargs[first_supported(inspect.signature(Trainer.__init__).parameters, "processing_class", "tokenizer")] = tokenizer
trainer = Trainer(**trainer_kwargs)
trainer.train()

## Step 6: read the loss curves

- **train loss** should fall from epoch to epoch (here it is the average over the epoch, with dropout on).
- **validation loss** is measured on held-out examples that use *different* policy passages.

If validation loss stops falling or **rises while training loss keeps falling**, the adapter is starting to memorize the small training set (overfitting). Then use fewer epochs, a lower learning rate, a smaller rank, or more varied data. Falling loss alone does not prove the model behaves well; that is what Step 7 checks.

In [ ]:
history = {}
for entry in trainer.state.log_history:
    if "epoch" not in entry:
        continue
    row = history.setdefault(round(float(entry["epoch"]), 3), {})
    if "eval_loss" in entry:
        row["val"] = entry["eval_loss"]
    elif "loss" in entry:
        row["train"] = entry["loss"]

print("epoch | train_loss | val_loss")
for epoch in sorted(history):
    row = history[epoch]
    print(f"{epoch:5.2f} | {row.get('train', float('nan')):10.4f} | {row.get('val', float('nan')):8.4f}")

try:
    import matplotlib.pyplot as plt

    epochs = sorted(history)
    plt.plot(epochs, [history[e].get("train", float("nan")) for e in epochs], marker="o", label="train loss")
    plt.plot(epochs, [history[e].get("val", float("nan")) for e in epochs], marker="o", label="validation loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.legend()
    plt.show()
except ImportError:
    print("(matplotlib is not installed; the table above shows the same numbers)")

## Step 7: evaluate base vs tuned

We generate answers (greedy decoding) for the held-out validation prompts and for the 10 generic probe prompts, once with the adapter switched off (**base**) and once with it on (**tuned**), and report:

- **JSON valid**: the whole output is one JSON object.
- **schema compliant**: exactly the keys `answer`, `tone` (= "formal"), `sources` (list of strings), `needs_escalation` (boolean).
- **escalation on unanswerable**: on validation examples whose context lacks the answer, did the model set `needs_escalation` to `true`?
- **general probe**: does the tuned model still answer simple general-knowledge questions (a check for catastrophic forgetting)?

The validation set is tiny (about 24 examples, 7 unanswerable), so treat differences as a smoke signal, not a benchmark.

In [ ]:
REQUIRED = ("answer", "tone", "sources", "needs_escalation")


def strict_json(text):
    try:
        obj = json.loads(text.strip())
    except ValueError:
        return None
    return obj if isinstance(obj, dict) else None


def schema_ok(obj):
    return (
        obj is not None
        and set(obj) == set(REQUIRED)
        and isinstance(obj["answer"], str) and obj["answer"].strip() != ""
        and obj["tone"] == "formal"
        and isinstance(obj["sources"], list) and all(isinstance(s, str) for s in obj["sources"])
        and isinstance(obj["needs_escalation"], bool)
    )


def generate(list_of_messages, max_new_tokens):
    device = next(model.parameters()).device
    model.eval()
    prompts = [encode_prompt(m) for m in list_of_messages]
    outputs = []
    for start in range(0, len(prompts), 8):
        batch = prompts[start:start + 8]
        longest = max(len(p) for p in batch)
        input_ids = torch.full((len(batch), longest), pad_id, dtype=torch.long)
        attention = torch.zeros((len(batch), longest), dtype=torch.long)
        for row, ids in enumerate(batch):                      # left padding for generation
            input_ids[row, longest - len(ids):] = torch.tensor(ids)
            attention[row, longest - len(ids):] = 1
        with torch.inference_mode():
            out = model.generate(
                input_ids=input_ids.to(device), attention_mask=attention.to(device),
                max_new_tokens=max_new_tokens, do_sample=False,
                temperature=None, top_p=None, top_k=None, repetition_penalty=1.0, pad_token_id=pad_id,
            )
        outputs += [tokenizer.decode(o[longest:], skip_special_tokens=True) for o in out.cpu()]
    return outputs


val_prompts = [r["messages"][:-1] for r in val_rows]
gold = [json.loads(r["messages"][-1]["content"]) for r in val_rows]
probe_prompts = [[{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": r["prompt"]}] for r in probe_rows]


def probe_pass(text, expected):
    """Same rule as common.py: case-insensitive substring, but a purely numeric
    expectation must not be part of a longer number (so "7" does not match "17")."""
    options = [expected] if isinstance(expected, str) else list(expected)
    for option in options:
        if option.isdigit():
            if re.search(rf"(?<!\d){re.escape(option)}(?!\d)", text):
                return True
        elif option.lower() in text.lower():
            return True
    return False


def score(val_out, probe_out):
    parsed = [strict_json(o) for o in val_out]
    unans = [i for i, g in enumerate(gold) if g["needs_escalation"]]
    return {
        "json valid": sum(p is not None for p in parsed) / len(parsed),
        "schema compliant": sum(schema_ok(p) for p in parsed) / len(parsed),
        "escalation on unanswerable": (sum(bool(parsed[i]) and parsed[i].get("needs_escalation") is True for i in unans) / len(unans)) if unans else float("nan"),
        "general probe pass": sum(probe_pass(o, r["expected"]) for o, r in zip(probe_out, probe_rows)) / len(probe_rows),
    }


with model.disable_adapter():          # adapter off = the original base model
    base_scores = score(generate(val_prompts, 220), generate(probe_prompts, 64))
tuned_out = generate(val_prompts, 220)
tuned_scores = score(tuned_out, generate(probe_prompts, 64))

print(f"{'metric':30s}{'base':>10s}{'tuned':>10s}")
for key in base_scores:
    print(f"{key:30s}{100 * base_scores[key]:9.1f}%{100 * tuned_scores[key]:9.1f}%")
print("\nSample tuned output:\n" + tuned_out[0])

## Step 8: save the adapter

Only the small adapter is saved (`adapter_config.json` + `adapter_model.safetensors`), not the base model. To use it later, load the same base model and attach the adapter with `PeftModel.from_pretrained` (that is what `evaluate.py` does), or run `merge_adapter.py` to create a standalone merged model.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(OUT_DIR))
print("Saved:", sorted(p.name for p in OUT_DIR.iterdir()))

try:  # Colab: download the adapter as a zip file
    import shutil
    from google.colab import files
    shutil.make_archive("lora-adapter", "zip", OUT_DIR)
    files.download("lora-adapter.zip")
except ImportError:
    pass

## Before you rely on this

- The hyper-parameters (learning rate, rank, epochs, batch size) are **starting points**, not tuned values. Change one thing at a time and compare on the validation set.
- With about 70 training examples, **overfitting** is easy. Watch Step 6.
- **Catastrophic forgetting**: fine-tuning can degrade abilities you did not train on. The 10-prompt probe is a crude check, not a guarantee. Test on your own generic tasks too.
- **Evaluate on held-out data** that the model never saw (and, ideally, on real cases from your own domain) before trusting any number.
- Fine-tuning changes behavior. For facts that change, need citations, or need per-user access control, use retrieval (RAG) and keep the facts out of the weights.